# MCP and A2A Explained by Running Them (Ollama, local)

Two protocols that are often confused — and that solve **different** problems:

| | **MCP** — Model Context Protocol | **A2A** — Agent2Agent Protocol |
|---|---|---|
| Connects | an **agent ↔ its tools / data** (vertical) | an **agent ↔ another agent** (horizontal) |
| Analogy | USB-C for AI: plug any tool into any model | a phone call between independent colleagues |
| Created by | Anthropic (2024) | Google, now Linux Foundation (2025) |
| Discovery | server lists its `tools`, `resources`, `prompts` | remote agent publishes an **Agent Card** (`/.well-known/agent-card.json`) |
| Wire format | JSON-RPC 2.0 over **stdio** or HTTP | JSON-RPC 2.0 over **HTTP** |
| Core call | `tools/call` | `message/send` → returns a **Task** with **Artifacts** |
| Callee is | a dumb function / data source | another *intelligent*, opaque agent with its own LLM |

## The scenario: "Trip Concierge"
A traveller asks: *"I'm flying to London tomorrow. What's the weather like, and how much is 500 GBP in Indian rupees?"*

```
                         A2A (HTTP, JSON-RPC)                       MCP (stdio, JSON-RPC)
 User ─▶ Trip Concierge ──────────────▶ Weather Agent  ───────────▶ weather MCP server ─▶ Open-Meteo API
         (orchestrator)  ──────────────▶ Finance Agent  ───────────▶ finance MCP server ─▶ Frankfurter API
          LLM routes &          each agent is a separate process      each agent's tools are
          merges answers        with its own LLM + Agent Card         plugged in through MCP
```

* **MCP** is used *inside* each specialist: the agent's LLM discovers and calls tools through a standard interface.
* **A2A** is used *between* the concierge and the specialists: the concierge only sees an Agent Card and sends a message — it knows nothing about the specialist's LLM, prompts or tools.

Everything runs locally with Ollama and free key-less APIs (Open-Meteo, Frankfurter).

## 0. Setup

In [ ]:
%pip install -q "mcp<2" ollama fastapi uvicorn httpx requests

> We pin `mcp<2` because the FastMCP-style API used here is the widely documented 1.x line.

Start Ollama and pull a tool-capable model:
```bash
ollama pull llama3.2
```

In [ ]:
import os, sys, json, time, subprocess, asyncio
from pathlib import Path
import ollama

MODEL = "llama3.2"                      # any local tool-capable model
os.environ["OLLAMA_MODEL"] = MODEL      # child processes (our A2A agents) read this
SERVERS = Path("mcp_a2a_servers").resolve()
SERVERS.mkdir(exist_ok=True)

names = [m.model for m in ollama.list().models]
assert any(n.startswith(MODEL) for n in names), f"Run: ollama pull {MODEL}"
print("Ollama OK. Server files will be written to:", SERVERS)

---
# Part 1 — MCP: giving an agent tools through a standard interface

**The problem MCP solves:** without a standard, every (model × tool) pair needs custom glue code. With MCP, a tool provider writes **one MCP server**; any MCP-capable client (Claude Desktop, IDEs, *your own agent*) can use it.

An MCP server can expose three things:
* **Tools** — functions the model can call (`get_current_weather`)
* **Resources** — read-only data the app can load (`weather://about`)
* **Prompts** — reusable prompt templates (`packing_advice`)

### 1.1 Write a Weather MCP server
It's a normal Python file using the `FastMCP` helper. Type hints and docstrings become the tool's JSON schema and description — which is exactly what the LLM will read.

In [ ]:
%%writefile mcp_a2a_servers/weather_mcp_server.py
import requests
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("weather-server")

def _geocode(city: str):
    r = requests.get("https://geocoding-api.open-meteo.com/v1/search", params={"name": city, "count": 1}, timeout=10).json()
    return r["results"][0] if r.get("results") else None

@mcp.tool()
def get_current_weather(city: str) -> str:
    """Get the live current weather for a city: temperature (C), wind (km/h) and rain (mm). Example city: 'London'."""
    p = _geocode(city)
    if not p:
        return f"City '{city}' not found."
    w = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": p["latitude"], "longitude": p["longitude"],
        "current": "temperature_2m,wind_speed_10m,precipitation"}, timeout=10).json()["current"]
    return f"{p['name']}, {p.get('country', '')}: {w['temperature_2m']}C, wind {w['wind_speed_10m']} km/h, rain {w['precipitation']} mm"

@mcp.tool()
def get_forecast(city: str, days: int = 3) -> str:
    """Get a daily forecast (min/max temperature C, chance of rain %) for the next `days` days (1-7)."""
    p = _geocode(city)
    if not p:
        return f"City '{city}' not found."
    days = max(1, min(int(days), 7))
    d = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": p["latitude"], "longitude": p["longitude"], "timezone": "auto", "forecast_days": days,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max"}, timeout=10).json()["daily"]
    rows = [f"{d['time'][i]}: {d['temperature_2m_min'][i]}-{d['temperature_2m_max'][i]}C, rain chance {d['precipitation_probability_max'][i]}%"
            for i in range(len(d["time"]))]
    return f"Forecast for {p['name']}:\n" + "\n".join(rows)

@mcp.resource("weather://about")
def about() -> str:
    """Where this server's data comes from."""
    return "Data: Open-Meteo (open-meteo.com), free and key-less. Temperatures in Celsius, wind in km/h."

@mcp.prompt()
def packing_advice(city: str) -> str:
    """A reusable prompt template that asks for packing advice."""
    return f"Check the weather in {city} and tell me what to pack, in 3 short bullet points."

if __name__ == "__main__":
    mcp.run(transport="stdio")      # NOTE: stdout is the protocol channel, so never print() in an MCP stdio server

### 1.2 Look at the actual protocol (raw JSON-RPC over stdio)
Before using any SDK, let's talk to the server by hand. MCP is just **newline-delimited JSON-RPC 2.0** on the server's stdin/stdout.
The handshake is always: `initialize` → `notifications/initialized` → then ask for `tools/list` or call `tools/call`.

In [ ]:
def raw_mcp_conversation(script, messages):
    p = subprocess.Popen([sys.executable, script], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                         stderr=subprocess.DEVNULL, text=True)
    try:
        for m in messages:
            p.stdin.write(json.dumps(m) + "\n"); p.stdin.flush()
            kind = "request " if "id" in m else "notification"
            print(f"CLIENT -> SERVER ({kind}): {json.dumps(m)}")
            if "id" in m:
                print(f"SERVER -> CLIENT (response): {p.stdout.readline().strip()[:700]}\n")
            else:
                print()
    finally:
        p.kill()

raw_mcp_conversation(str(SERVERS / "weather_mcp_server.py"), [
    {"jsonrpc": "2.0", "id": 1, "method": "initialize",
     "params": {"protocolVersion": "2025-06-18", "capabilities": {}, "clientInfo": {"name": "raw-client", "version": "0.1"}}},
    {"jsonrpc": "2.0", "method": "notifications/initialized"},
    {"jsonrpc": "2.0", "id": 2, "method": "tools/list"},
    {"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "get_current_weather", "arguments": {"city": "London"}}},
])

**What to notice**
* `tools/list` returned each tool's **name, description and JSON Schema** — generated from our type hints. This is the "menu" an LLM reads.
* `tools/call` took a tool name plus arguments and returned a `content` list. The client never needed to know *how* the weather is fetched.

### 1.3 Use the official MCP client SDK
The SDK handles the handshake and message framing. We can ask the server for all three capability types.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def inspect_server(script):
    params = StdioServerParameters(command=sys.executable, args=[script])
    with open(os.devnull, "w") as quiet:                       # keep server logs out of the notebook
        async with stdio_client(params, errlog=quiet) as (read, write):
            async with ClientSession(read, write) as session:
                init = await session.initialize()
                print(f"Connected to: {init.serverInfo.name}  (protocol {init.protocolVersion})\n")
                print("TOOLS:")
                for t in (await session.list_tools()).tools:
                    print(f"  - {t.name}: {t.description.splitlines()[0]}\n      args: {list(t.inputSchema['properties'])}")
                print("RESOURCES:", [str(r.uri) for r in (await session.list_resources()).resources])
                print("   weather://about ->", (await session.read_resource("weather://about")).contents[0].text)
                print("PROMPTS:", [p.name for p in (await session.list_prompts()).prompts])
                print("   rendered ->", (await session.get_prompt("packing_advice", {"city": "Oslo"})).messages[0].content.text)
                res = await session.call_tool("get_forecast", {"city": "Oslo", "days": 2})
                print("\nCALL get_forecast('Oslo', 2) ->\n" + res.content[0].text)

await inspect_server(str(SERVERS / "weather_mcp_server.py"))

### 1.4 Let an LLM use the MCP server

Here is the key idea: our agent code contains **no weather-specific logic**. It
1. connects to *any* MCP server and lists its tools,
2. hands those tool schemas to the LLM,
3. executes whatever tool calls the LLM makes **through MCP**, and feeds results back.

We save this loop as a module (`mcp_agent.py`) because our A2A agents in Part 2 will reuse it.

#### 🤖 Agent profile — MCP-powered assistant
- **Goal:** answer questions using whatever tools an MCP server provides, without hard-coding any tool.
- **Input:** a system prompt, the user's question, and the path of an MCP server to connect to.
- **Tools:** discovered at runtime with `tools/list`; executed with `tools/call`.
- **Output:** the LLM's final natural-language answer, grounded in live tool results.
- **Why it matters:** swap the server and the same agent gains new abilities (we do exactly that in 1.5).

In [ ]:
%%writefile mcp_a2a_servers/mcp_agent.py
import os, sys
from contextlib import asynccontextmanager
import ollama
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MODEL = os.environ.get("OLLAMA_MODEL", "llama3.2")

@asynccontextmanager
async def mcp_session(server_script):
    """Spawn an MCP server over stdio and yield an initialised client session."""
    params = StdioServerParameters(command=sys.executable, args=[server_script])
    with open(os.devnull, "w") as quiet:
        async with stdio_client(params, errlog=quiet) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session

def to_ollama_tools(mcp_tools):
    """MCP tool definitions already contain a JSON Schema, which is what Ollama's tool-calling expects."""
    return [{"type": "function",
             "function": {"name": t.name, "description": t.description or "", "parameters": t.inputSchema}}
            for t in mcp_tools]

async def run_with_mcp(server_script, system_prompt, question, label="agent", model=MODEL, max_steps=5):
    client = ollama.AsyncClient()
    async with mcp_session(server_script) as session:
        tools = to_ollama_tools((await session.list_tools()).tools)
        print(f"[{label}] MCP tools available: {[t['function']['name'] for t in tools]}", flush=True)
        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": question}]
        nudged = False
        for _ in range(max_steps):
            msg = (await client.chat(model=model, messages=messages, tools=tools, options={"temperature": 0})).message
            messages.append(msg)
            if not msg.tool_calls:
                if not nudged and not any(m["role"] == "tool" for m in messages if isinstance(m, dict)):
                    # Small models sometimes answer without using a tool. Ollama has no "tool required" switch,
                    # so we emulate it: remind the model once and let it try again.
                    nudged = True
                    print(f"[{label}] no tool call yet -> reminding the model to use its tools", flush=True)
                    messages.append({"role": "user", "content": "You must call one of your tools to get real data before answering."})
                    continue
                return msg.content
            for call in msg.tool_calls:
                name, args = call.function.name, dict(call.function.arguments)
                print(f"[{label}] LLM -> MCP tools/call {name}({args})", flush=True)
                result = await session.call_tool(name, args)
                text = "\n".join(c.text for c in result.content if getattr(c, "type", "") == "text")
                print(f"[{label}] MCP result: {text[:150]!r}", flush=True)
                messages.append({"role": "tool", "tool_name": name, "content": text})
        return "(stopped: max steps reached)"

In [ ]:
sys.path.insert(0, str(SERVERS))
import importlib, mcp_agent
importlib.reload(mcp_agent)
from mcp_agent import run_with_mcp

answer = await run_with_mcp(
    str(SERVERS / "weather_mcp_server.py"),
    "You are a weather assistant. Use the tools to get real data, then answer in 2 short lines.",
    "Is it raining in Mumbai right now, and what is the 3 day outlook?",
    label="MCP assistant")
print("\nANSWER:", answer)

### 1.5 Swap the server, gain a new skill (same agent code)
Now a **Finance MCP server** with currency tools. The client code above does not change at all — only the server path.

In [ ]:
%%writefile mcp_a2a_servers/finance_mcp_server.py
import requests
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("finance-server")
API = "https://api.frankfurter.dev/v1/latest"

@mcp.tool()
def get_exchange_rate(base: str, target: str) -> str:
    """Get the live exchange rate: how many units of `target` currency equal 1 unit of `base`. Use ISO codes like USD, INR, GBP."""
    r = requests.get(API, params={"base": base.upper(), "symbols": target.upper()}, timeout=10).json()
    if "rates" not in r:
        return f"No rate available for {base}->{target}."
    return f"1 {base.upper()} = {r['rates'][target.upper()]} {target.upper()} (ECB rate, {r['date']})"

@mcp.tool()
def convert_currency(amount: float, base: str, target: str) -> str:
    """Convert an amount of money from `base` currency to `target` currency using the live rate. Use ISO codes like USD, INR, GBP."""
    r = requests.get(API, params={"base": base.upper(), "symbols": target.upper()}, timeout=10).json()
    if "rates" not in r:
        return f"No rate available for {base}->{target}."
    rate = r["rates"][target.upper()]
    return f"{amount} {base.upper()} = {amount * rate:,.2f} {target.upper()} (rate {rate}, {r['date']})"

@mcp.resource("finance://about")
def about() -> str:
    """Where this server's data comes from."""
    return "Data: Frankfurter API (European Central Bank reference rates), updated on working days."

if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
answer = await run_with_mcp(
    str(SERVERS / "finance_mcp_server.py"),
    "You are a currency assistant. Use the tools for real rates, then answer in 1-2 short lines.",
    "How many Indian rupees is 500 British pounds?",
    label="MCP assistant")
print("\nANSWER:", answer)

**Part 1 takeaway:** MCP standardised *how an agent finds and calls tools*. One client loop, many servers. But a single agent with many tools still has limits — which leads to A2A.

---
# Part 2 — A2A: independent agents collaborating over HTTP

**The problem A2A solves:** agents are built by different teams, frameworks and vendors. Each is an **opaque** service with its own LLM, memory and tools. A2A gives them a common way to **discover** each other and **exchange tasks**, without exposing internals.

### Core A2A concepts
| Concept | What it is |
|---|---|
| **Agent Card** | JSON "business card" at `/.well-known/agent-card.json`: name, description, URL, **skills**, supported input/output modes |
| **Message** | one turn of conversation: `role` (`user`/`agent`) + `parts` (text, files, data) |
| **Task** | the unit of work with a lifecycle: `submitted → working → completed / failed / input-required` |
| **Artifact** | the output produced by a task (e.g. the answer text) |
| **`message/send`** | JSON-RPC method: send a message, receive the Task back |

> **Implementation note:** to make the protocol visible we implement the A2A server side ourselves (≈60 lines of FastAPI) following the spec's JSON shapes. In production you would use the official `a2a-sdk` package, which adds streaming (`message/stream`), push notifications and authentication.

#### 🤖 Agent profiles
**Weather Agent** (A2A server, port 8801)
- **Goal:** be the team's weather expert. **Input:** a natural-language weather question received via A2A.
- **Tools:** the weather MCP server (`get_current_weather`, `get_forecast`) — discovered through MCP, not hard-coded.
- **Output:** a Task whose artifact is a short, data-grounded weather summary.
- **Skill advertised on its Agent Card:** `weather-lookup`.

**Finance Agent** (A2A server, port 8802)
- **Goal:** be the team's currency expert. **Input:** a natural-language money / exchange-rate question via A2A.
- **Tools:** the finance MCP server (`get_exchange_rate`, `convert_currency`).
- **Output:** a Task whose artifact is a short answer with the live rate.
- **Skill advertised on its Agent Card:** `currency-conversion`.

**Trip Concierge** (orchestrator = A2A *client*, runs in this notebook)
- **Goal:** answer a traveller's multi-part question by delegating to the right specialists.
- **Input:** the user's free-text question and the Agent Cards of the agents it discovered.
- **Tools:** none of its own — only A2A calls to the other agents, plus an LLM to route and to merge answers.
- **Output:** one friendly combined answer.

### 2.1 Write the generic A2A agent server
One file runs *any* agent; behaviour comes from a small config file. Look for the three A2A pieces: the **Agent Card** endpoint, the **`message/send`** JSON-RPC method, and the **Task** object it returns. Inside, it uses the MCP loop from Part 1 — that is where A2A and MCP meet.

In [ ]:
%%writefile mcp_a2a_servers/a2a_agent_server.py
import sys, json, uuid, datetime
import uvicorn
from fastapi import FastAPI, Request
from mcp_agent import run_with_mcp

cfg = json.load(open(sys.argv[1]))
URL = f"http://127.0.0.1:{cfg['port']}/"

# ---- 1) AGENT CARD: how other agents discover what this agent can do
AGENT_CARD = {
    "name": cfg["name"],
    "description": cfg["description"],
    "url": URL,
    "version": "1.0.0",
    "protocolVersion": "0.3.0",
    "capabilities": {"streaming": False, "pushNotifications": False},
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "skills": cfg["skills"],
}

app = FastAPI(title=cfg["name"])
TASKS = {}          # in-memory task store: task_id -> Task

def now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

@app.get("/.well-known/agent-card.json")
@app.get("/.well-known/agent.json")            # older spec versions used this path
def agent_card():
    return AGENT_CARD

def rpc_error(rid, code, message):
    return {"jsonrpc": "2.0", "id": rid, "error": {"code": code, "message": message}}

# ---- 2) JSON-RPC endpoint: message/send and tasks/get
@app.post("/")
async def jsonrpc(request: Request):
    body = await request.json()
    rid, method, params = body.get("id"), body.get("method"), body.get("params", {})

    if method == "message/send":
        message = params["message"]
        question = " ".join(p["text"] for p in message["parts"] if p.get("kind") == "text")
        task_id, context_id = str(uuid.uuid4()), message.get("contextId") or str(uuid.uuid4())
        print(f"[{cfg['name']}] A2A message/send received -> task {task_id[:8]}: {question!r}", flush=True)
        try:
            # ---- 3) The agent does its work: LLM + tools discovered through MCP
            answer = await run_with_mcp(cfg["mcp_server"], cfg["system_prompt"], question, label=cfg["name"])
            task = {"kind": "task", "id": task_id, "contextId": context_id,
                    "status": {"state": "completed", "timestamp": now()},
                    "artifacts": [{"artifactId": str(uuid.uuid4()), "name": "answer",
                                   "parts": [{"kind": "text", "text": answer}]}],
                    "history": [message]}
        except Exception as e:
            task = {"kind": "task", "id": task_id, "contextId": context_id,
                    "status": {"state": "failed", "timestamp": now(),
                               "message": {"role": "agent", "messageId": str(uuid.uuid4()),
                                           "parts": [{"kind": "text", "text": f"Error: {e}"}]}},
                    "history": [message]}
        TASKS[task_id] = task
        print(f"[{cfg['name']}] task {task_id[:8]} -> {task['status']['state']}", flush=True)
        return {"jsonrpc": "2.0", "id": rid, "result": task}

    if method == "tasks/get":
        task = TASKS.get(params.get("id"))
        return {"jsonrpc": "2.0", "id": rid, "result": task} if task else rpc_error(rid, -32001, "Task not found")

    return rpc_error(rid, -32601, f"Method not found: {method}")

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=cfg["port"], log_level="warning")

### 2.2 Configure and start the two specialist agents
Each runs as its **own process** (as if owned by a different team) on its own port.

In [ ]:
WEATHER_CFG = {
    "name": "WeatherAgent", "port": 8801,
    "description": "Provides live weather conditions and short-term forecasts for any city in the world.",
    "mcp_server": str(SERVERS / "weather_mcp_server.py"),
    "system_prompt": "You are a Weather Agent. ALWAYS call a tool to get real data and never ask the user questions back. Use get_forecast for 'tomorrow' or multi-day outlooks (mention every day returned) and get_current_weather for right now. Once a tool has returned data, answer directly from it and do not call more tools. Answer in at most 4 short lines.",
    "skills": [{"id": "weather-lookup", "name": "Weather lookup",
                "description": "Current weather and multi-day forecast for a city.",
                "tags": ["weather", "forecast", "temperature", "rain"],
                "examples": ["What's the weather in Paris?", "Will it rain in Tokyo this week?"]}],
}
FINANCE_CFG = {
    "name": "FinanceAgent", "port": 8802,
    "description": "Provides live currency exchange rates and converts amounts between currencies.",
    "mcp_server": str(SERVERS / "finance_mcp_server.py"),
    "system_prompt": "You are a Finance Agent. ALWAYS call a tool to get real rates and never ask the user questions back. Once a tool has returned data, answer directly from it and do not call more tools. Answer in at most 2 short lines.",
    "skills": [{"id": "currency-conversion", "name": "Currency conversion",
                "description": "Exchange rates and amount conversion between currencies (ISO codes).",
                "tags": ["currency", "exchange rate", "money", "convert"],
                "examples": ["How much is 100 USD in EUR?", "What is the GBP to INR rate?"]}],
}
for cfg in (WEATHER_CFG, FINANCE_CFG):
    (SERVERS / f"{cfg['name']}.json").write_text(json.dumps(cfg, indent=2))

import httpx
PROCS = {}

def stop_agents():
    for p in PROCS.values():
        p.terminate()
    PROCS.clear()

def start_agent(cfg):
    log = open(SERVERS / f"{cfg['name']}.log", "w")
    p = subprocess.Popen([sys.executable, "-u", "a2a_agent_server.py", f"{cfg['name']}.json"],
                         cwd=SERVERS, stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
    PROCS[cfg["name"]] = p
    url = f"http://127.0.0.1:{cfg['port']}/.well-known/agent-card.json"
    for _ in range(60):                         # wait until the server answers
        try:
            if httpx.get(url, timeout=1).status_code == 200:
                print(f"started {cfg['name']} on port {cfg['port']} (pid {p.pid})")
                return
        except Exception:
            time.sleep(0.5)
    raise RuntimeError(f"{cfg['name']} did not start; see {SERVERS / (cfg['name'] + '.log')}")

stop_agents()
start_agent(WEATHER_CFG)
start_agent(FINANCE_CFG)

### 2.3 Discovery: read an Agent Card
Anyone can `GET` the card — no SDK, no shared code. This is how the concierge learns **who exists and what they can do**.

In [ ]:
card = httpx.get("http://127.0.0.1:8801/.well-known/agent-card.json").json()
print(json.dumps(card, indent=2))

### 2.4 The raw A2A call (what `message/send` really looks like)
One HTTP POST carrying a JSON-RPC request. The reply is a **Task** — with a `status.state` and the answer inside an **artifact**.

In [ ]:
import uuid
request_body = {
    "jsonrpc": "2.0", "id": 1, "method": "message/send",
    "params": {"message": {"kind": "message", "role": "user", "messageId": str(uuid.uuid4()),
                           "parts": [{"kind": "text", "text": "What's the weather in Paris right now?"}]}},
}
print("REQUEST ->\n" + json.dumps(request_body, indent=2))

async with httpx.AsyncClient(timeout=180) as client:
    response = (await client.post("http://127.0.0.1:8801/", json=request_body)).json()

print("\nRESPONSE <-\n" + json.dumps(response, indent=2))
print("\nTask state:", response["result"]["status"]["state"])
print("Answer    :", response["result"]["artifacts"][0]["parts"][0]["text"])

Notice what the **caller did not need to know**: that the Weather Agent uses an LLM, which model, that it calls an MCP server, or which weather API is behind it. That opacity is the point of A2A.
You can see the agent's *private* side — its MCP tool calls — in its own log file:

In [ ]:
print((SERVERS / "WeatherAgent.log").read_text())

### 2.5 The orchestrator: Trip Concierge discovers, delegates, merges
The concierge is an **A2A client**. Its steps:
1. **Discover** — fetch every agent's card.
2. **Route** — an LLM (using JSON-schema–constrained output) decides which agents are needed and what to ask each (copying the user's own wording so no detail is lost). It reads only the cards' descriptions and skills.
3. **Delegate** — send an A2A `message/send` to each chosen agent (concurrently).
4. **Merge** — an LLM combines the returned artifacts into one answer.

In [ ]:
AGENT_URLS = ["http://127.0.0.1:8801", "http://127.0.0.1:8802"]

def discover(urls):
    cards = {}
    for u in urls:
        c = httpx.get(f"{u}/.well-known/agent-card.json", timeout=10).json()
        cards[c["name"]] = c
    return cards

async def a2a_send(url, text):
    body = {"jsonrpc": "2.0", "id": str(uuid.uuid4()), "method": "message/send",
            "params": {"message": {"kind": "message", "role": "user", "messageId": str(uuid.uuid4()),
                                   "parts": [{"kind": "text", "text": text}]}}}
    async with httpx.AsyncClient(timeout=180) as client:
        reply = (await client.post(url, json=body)).json()
    if "error" in reply:
        raise RuntimeError(reply["error"]["message"])
    task = reply["result"]
    if task["status"]["state"] != "completed":
        raise RuntimeError(f"task {task['status']['state']}")
    return task["artifacts"][0]["parts"][0]["text"], task

def route(question, cards):
    """Ask the LLM which agents are needed and what to ask each (schema-constrained output; names are validated)."""
    directory = "\n".join(f"- {n}: {c['description']} Skills: {[s['name'] for s in c['skills']]}" for n, c in cards.items())
    schema = {"type": "object", "required": ["delegations"], "properties": {"delegations": {"type": "array", "items": {
        "type": "object", "required": ["agent", "task"],
        "properties": {"agent": {"type": "string", "enum": list(cards)}, "task": {"type": "string"}}}}}}
    resp = ollama.chat(model=MODEL, format=schema, options={"temperature": 0}, messages=[
        {"role": "system", "content": "You are a router. For each agent that is needed, output the agent and its task. The task MUST be copied "
                                      "from the user's own words (the clause that concerns that agent's skills), keeping every detail such as "
                                      "place, time range, amounts and currencies. Do not paraphrase, shorten or invent. "
                                      f"Only include agents that are needed.\nAgents:\n{directory}"},
        {"role": "user", "content": question}])
    plan = json.loads(resp.message.content)["delegations"]
    return [d for d in plan if d["agent"] in cards]

async def concierge(question):
    print("USER:", question, "\n")
    cards = discover(AGENT_URLS)
    print("1) DISCOVERED:", {n: [s["id"] for s in c["skills"]] for n, c in cards.items()})
    plan = route(question, cards)
    print("2) ROUTED   :", json.dumps(plan))
    results = await asyncio.gather(*[a2a_send(cards[d["agent"]]["url"], d["task"]) for d in plan])
    print("3) A2A REPLIES:")
    for d, (text, task) in zip(plan, results):
        print(f"   - {d['agent']} [task {task['id'][:8]} {task['status']['state']}]: {text}")
    notes = "\n".join(f"{d['agent']}: {text}" for d, (text, _) in zip(plan, results))
    final = ollama.chat(model=MODEL, options={"temperature": 0}, messages=[
        {"role": "system", "content": "You are a Trip Concierge. Combine the specialists' answers into one short, friendly reply. "
                                      "Use ONLY the information given; do not add facts."},
        {"role": "user", "content": f"Question: {question}\n\nSpecialist answers:\n{notes}"}]).message.content
    print("\n4) FINAL ANSWER:\n" + final)
    return final

_ = await concierge("I'm flying to London tomorrow. What's the weather like there, and how much is 500 GBP in Indian rupees?")

**A narrower question** — the router should involve only one agent (a benefit of discovery-based routing):

In [ ]:
_ = await concierge("Will it rain in Tokyo over the next 3 days?")

Peek at the Finance Agent's private log: its LLM→MCP calls happened on the other side of the A2A boundary.

In [ ]:
print((SERVERS / "FinanceAgent.log").read_text())

In [ ]:
stop_agents()       # shut down the two agent servers
print("agents stopped")

---
# Part 3 — How MCP and A2A fit together

```
        ┌────────────────────────── A2A (agent ↔ agent) ──────────────────────────┐
        │                                                                          │
  Trip Concierge ─────────▶ Weather Agent                       Finance Agent ◀─────┘
                               │  MCP (agent ↔ tools)               │  MCP
                               ▼                                    ▼
                      weather MCP server                    finance MCP server
                               │                                    │
                          Open-Meteo API                      Frankfurter API
```

| Question | MCP | A2A |
|---|---|---|
| What does it connect? | an agent to **tools and data** | an agent to **another agent** |
| Is the other side intelligent? | No — it executes a function | Yes — it reasons, may ask follow-ups, may take long |
| What do you see of the other side? | its full tool list and schemas (transparent) | only its Agent Card and results (**opaque**) |
| Typical unit of work | a single call → a result | a **Task** with a lifecycle and artifacts |
| Transport | stdio (local) or HTTP | HTTP |
| Use it when | you want one agent to use many tools/data sources consistently | agents from different teams / frameworks / vendors must collaborate |

**Rules of thumb**
* Need a **capability** (search, database, calendar)? → expose it as an **MCP server**.
* Need a **collaborator** with its own judgement (a travel agent, a legal reviewer)? → expose it as an **A2A agent**.
* They compose: an A2A agent usually uses MCP internally, exactly as in this notebook.

**Security notes (even for a demo):** treat tool output and remote-agent replies as *untrusted data*; authenticate A2A endpoints in production (the Agent Card declares supported auth schemes); only run MCP servers you trust, since they execute code with your privileges.

### Ideas to extend
1. Add a third specialist (e.g. an **Attractions Agent** backed by a Wikipedia MCP server) and watch the router use it.
2. Replace our hand-written A2A server with the official `a2a-sdk` and add `message/stream` for streaming answers.
3. Run an MCP server over **streamable HTTP** instead of stdio so it can live on another machine.
4. Put a LangGraph agent behind the A2A interface — the concierge won't notice any difference.
5. Make the Finance Agent return `input-required` when the currency is ambiguous, and let the concierge ask the user.